# California County Climate Report

Generates a one-page, report-style climate summary for any of California's
58 counties using `climakitae.ClimateData().get()` and the reusable
rendering components in `climakitae.visualize`.

**What you get**

| Panel | Description |
|---|---|
| Stat cards | Three headline numbers (avg-high rise, heat-wave frequency, extra hot days) |
| Summary table | Metric × period table: Historic · Near · Mid · Late century |
| Bar chart | Countywide 1-in-10-yr extreme heat threshold: historical vs late-century |

**Data**: WRF dynamical downscaling — daily `t2max` / `t2min`, 9 km grid
(`d02`), converted to °F, clipped to the selected county.

**Periods (all global warming levels, ±15 yr window per ensemble member)**

| Period | GWL |
|---|---|
| Historic baseline | 0.8 °C |
| Near-century      | 1.5 °C |
| Mid-century       | 2.0 °C |
| Late-century      | 3.0 °C |

> **Caveat** — The 1-in-10-yr threshold is an annual-max-quantile estimate.
> For rigorous return-period analysis, use the `metric_calc one_in_x`
> processor (GEV fit), preferably on LOCA2 for better tail estimates.
> WRF ensemble size is smaller than LOCA2; treat summaries as central
> tendencies, not the full uncertainty range.

## 0 · Install `climakitae.visualize`

Until [cal-adapt/climakitae#765](https://github.com/cal-adapt/climakitae/pull/765)
is merged, install directly from the feature branch:

In [1]:
!uv pip install -q 'climakitae @ git+https://github.com/cal-adapt/climakitae.git@feature/report-visualize'

## 1 · Imports

In [2]:
from __future__ import annotations

import warnings

import ipywidgets as widgets
import pandas as pd
from IPython.display import display

from climakitae.new_core.user_interface import ClimateData
from climakitae.visualize import (
    PeriodInputs,
    build_report_figure,
    compute_report_metrics,
)

warnings.filterwarnings("ignore", category=FutureWarning)

## 2 · Choose county and thresholds

In [ ]:
CA_COUNTIES = [
    "Alameda County", "Alpine County", "Amador County", "Butte County",
    "Calaveras County", "Colusa County", "Contra Costa County", "Del Norte County",
    "El Dorado County", "Fresno County", "Glenn County", "Humboldt County",
    "Imperial County", "Inyo County", "Kern County", "Kings County",
    "Lake County", "Lassen County", "Los Angeles County", "Madera County",
    "Marin County", "Mariposa County", "Mendocino County", "Merced County",
    "Modoc County", "Mono County", "Monterey County", "Napa County",
    "Nevada County", "Orange County", "Placer County", "Plumas County",
    "Riverside County", "Sacramento County", "San Benito County",
    "San Bernardino County", "San Diego County", "San Francisco County",
    "San Joaquin County", "San Luis Obispo County", "San Mateo County",
    "Santa Barbara County", "Santa Clara County", "Santa Cruz County",
    "Shasta County", "Sierra County", "Siskiyou County", "Solano County",
    "Sonoma County", "Stanislaus County", "Sutter County", "Tehama County",
    "Trinity County", "Tulare County", "Tuolumne County", "Ventura County",
    "Yolo County", "Yuba County",
]

county_w = widgets.Dropdown(
    options=CA_COUNTIES,
    value="Los Angeles County",
    description="County:",
    layout=widgets.Layout(width="380px"),
    style={"description_width": "initial"},
)
hot_day_w = widgets.IntSlider(
    value=90, min=80, max=115, step=1,
    description="Hot-day threshold (\u00b0F):",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="420px"),
)
heatwave_w = widgets.IntSlider(
    value=4, min=2, max=10, step=1,
    description="Heat-wave min consecutive days:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="420px"),
)

display(widgets.VBox([
    widgets.HTML("<b>Select county and thresholds, then run the cells below.</b>"),
    county_w,
    hot_day_w,
    heatwave_w,
]))

WARMING_LEVELS = [0.8, 1.5, 2.0, 3.0]
PERIOD_LABELS = {
    0.8: "Historic\n(GWL 0.8\u00b0C)",
    1.5: "Near-century\n(GWL 1.5\u00b0C)",
    2.0: "Mid-century\n(GWL 2.0\u00b0C)",
    3.0: "Late-century\n(GWL 3.0\u00b0C)",
}

## 3 · Resolve selections

Snapshot widget values into plain Python so the rest of the notebook is
deterministic when re-run end-to-end.

In [4]:
county: str = county_w.value
hot_day_threshold_F: int = int(hot_day_w.value)
heatwave_min_days: int = int(heatwave_w.value)
warming_levels: list[float] = WARMING_LEVELS

print(f"County                    : {county}")
print(f"Hot-day threshold         : >{hot_day_threshold_F} °F")
print(f"Heat-wave minimum days    : {heatwave_min_days}")
print(f"Warming levels            : {warming_levels} °C")

County                    : Los Angeles County
Hot-day threshold         : >90 °F
Heat-wave minimum days    : 4
Warming levels            : [1.5, 2.0] °C


## 4 · Fetch all warming-level windows

WRF daily `t2max` and `t2min`, 9 km grid, clipped to the selected county,
converted to °F. The historical baseline is GWL 0.8 °C; near/mid/late
century are 1.5, 2.0, and 3.0 °C respectively. climakitae selects a
±15-yr window around each ensemble member's crossing of each level.
*(Expect ~30 s – 3 min per level depending on data-store latency.)*

In [5]:
def _fetch_gwl(variable: str, level: float):
    """Fetch one WRF variable for a global warming level window."""
    return (
        ClimateData(verbosity=-1)
        .catalog("cadcat")
        .variable(variable)
        .activity_id("WRF")
        .institution_id("UCLA")
        .table_id("day")
        .grid_label("d02")
        .experiment_id("ssp370")
        .processes({
            "warming_level": {"warming_levels": [level]},
            "clip": county,
            "convert_units": "degF",
        })
        .get()
    )


gwl_data: dict[float, dict[str, object]] = {}
for level in warming_levels:
    print(f"Fetching GWL {level} °C …")
    gwl_data[level] = {
        "tmax": _fetch_gwl("t2max", level),
        "tmin": _fetch_gwl("t2min", level),
    }

print("Done. Warming levels fetched:", list(gwl_data))

2026-05-18 12:28:05 - climakitae.new_core.processors.filter_unadjusted_models - WARNING - 

Your query selected models that do not have a-priori bias adjustment. 
These models have been removed from the returned query. 
To include them, please add the following processor to your query: 
ClimateData().processes('filter_unadjusted_models': 'no')

2026-05-18 12:28:11 - climakitae.new_core.processors.filter_unadjusted_models - WARNING - 

Your query selected models that do not have a-priori bias adjustment. 
These models have been removed from the returned query. 
To include them, please add the following processor to your query: 
ClimateData().processes('filter_unadjusted_models': 'no')

t2max shape: {'sim': 5, 'y': 25, 'x': 22, 'time': 10950}
t2min shape: {'sim': 5, 'y': 25, 'x': 22, 'time': 10950}


## 5 · Compute report metrics

`compute_report_metrics` accepts an ordered dict of period label →
`PeriodInputs(tmax, tmin)` and returns a tidy `DataFrame` (metrics as rows,
periods as columns) that the renderers consume directly.

In [7]:
periods: dict[str, PeriodInputs] = {}
for level in warming_levels:
    periods[PERIOD_LABELS[level]] = PeriodInputs(
        tmax=gwl_data[level]["tmax"],
        tmin=gwl_data[level]["tmin"],
    )

summary_df = compute_report_metrics(
    periods,
    hot_day_threshold_F=hot_day_threshold_F,
    heatwave_min_days=heatwave_min_days,
    return_period_years=10,
)
summary_df

TypeError: float() argument must be a string or a real number, not 'method'

## 6 · Derive headline stat-card numbers

Three numbers that surface the most-cited changes:

In [ ]:
hist_col = summary_df.columns[0]   # e.g. 'Historic\n(1981-2010)'
proj_col = summary_df.columns[-1]  # last warming level

avg_high_row = next(r for r in summary_df.index if "Average High" in r)
hw_row       = next(r for r in summary_df.index if "Heat Waves" in r)
hot_row      = next(r for r in summary_df.index if "Hot Days" in r)

delta_high = summary_df.loc[avg_high_row, proj_col] - summary_df.loc[avg_high_row, hist_col]
hist_hw    = summary_df.loc[hw_row, hist_col]
proj_hw    = summary_df.loc[hw_row, proj_col]
extra_hot  = summary_df.loc[hot_row, proj_col] - summary_df.loc[hot_row, hist_col]

hw_label = f"{proj_hw / hist_hw:.1f}×" if hist_hw > 0 else f"+{proj_hw:.0f}"

stat_items = [
    (f"+{delta_high:.1f}°F",  f"Avg summer high vs historical ({proj_col.replace(chr(10), ' ')})"),
    (hw_label,                 "Heat-wave frequency vs historical"),
    (f"+{extra_hot:.0f}",      f"Extra hot days >{hot_day_threshold_F}°F per year"),
]

for val, caption in stat_items:
    print(f"  {val:>10}  {caption}")

## 7 · Build bars-chart input

Compares the countywide 1-in-10-yr extreme heat threshold across
historical and the most-extreme projection (last warming level).

In [ ]:
extreme_row = next(r for r in summary_df.index if "1-in-" in r)

bars_df = pd.DataFrame(
    {"Historical": [summary_df.loc[extreme_row, hist_col]],
     "Projection": [summary_df.loc[extreme_row, proj_col]]},
    index=[county],
)
bars_df

## 8 · Render the composite report figure

In [ ]:
fig = build_report_figure(
    title=f"{county.removesuffix(' County')} County — Extreme Heat Climate Report",
    subtitle="WRF · daily · GWL 0.8 °C (historic) vs 1.5 / 2.0 / 3.0 °C",
    stat_items=stat_items,
    summary_df=summary_df,
    bars_df=bars_df,
    bars_title="1-in-10-yr Daily Max Temperature (°F): Historic vs Late-century",
)
fig

## 9 · (Optional) Export to PNG / PDF

In [ ]:
# from pathlib import Path
#
# out_dir = Path("figures")
# out_dir.mkdir(exist_ok=True)
# slug = county.lower().replace(" ", "_")
#
# fig.savefig(out_dir / f"{slug}_climate_report.png", dpi=200, bbox_inches="tight")
# fig.savefig(out_dir / f"{slug}_climate_report.pdf",          bbox_inches="tight")
# print(f"Saved to {out_dir}/")